# Maia Corpus v8 — Tokenizer Validation & Pretraining Preparation

**Maia Lite 0.3.0 Small | Reproducible Colab notebook**

This notebook uses the completed scientific freeze (`freeze_v8_001`) and never changes the original audit database, configuration, corpus chunks, or freeze files. It implements: (1) manifest checks, (2) discovery of actual JSONL layout, (3) full-chunk tokenizer recount on a deterministic stratified sample, (4) optional tokenizer provenance evidence, (5) deterministic document-hash train/validation split, and (6) a bounded pilot for packing and training-data statistics.

**Release gates:** historical tokenizer identity cannot be proven from a matching count alone; the audit's counting convention must be confirmed. Cross-document near-duplicates, source licensing, and full-corpus leakage prevention require additional checks before production pretraining. **No full corpus materialization is started by this notebook.**

In [ ]:
# Colab: mount Drive (skip if already mounted)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; ensure the corpus paths are mounted locally.')

In [ ]:
# Dependencies: pinned versions should be recorded in the pilot report.
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'transformers>=4.44,<5', 'tokenizers>=0.19', 'tqdm'])

In [ ]:
import os, json, gzip, hashlib, sqlite3, random, re, platform
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone
import transformers, tokenizers
from transformers import AutoTokenizer

RUN = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001')
FREEZE = RUN / 'freeze_v8_001'
MANIFEST_PATH = FREEZE / 'manifest.json'
AUDIT_DB = RUN / 'quality_audit_full_with_tokenizer_v1/audit_state.sqlite3'
TOKENIZER_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
# The corpus chunks may be elsewhere; discovery below does not assume a fixed layout.
SEARCH_ROOTS = [RUN, RUN.parent.parent, RUN.parent.parent.parent]
OUTPUT = RUN / 'pretraining_prep_pilot_v1'
SEED = 20261026
SAMPLE_CHUNKS_PER_DATASET = 2  # 12 chunks total, approximately 6k documents
MAX_PILOT_DOCS = 1200
BLOCK_SIZE = 1024  # PILOT ONLY: must be matched to final architecture
VALIDATION_PERCENT = 1
WRITE_PILOT = True

assert MANIFEST_PATH.is_file() and AUDIT_DB.is_file() and TOKENIZER_DIR.is_dir()
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert manifest['audit_summary']['chunks'] == 21021
assert manifest['audit_summary']['real_tokens'] == 7590554035
assert manifest['audit_summary']['documents'] == 10101231
print('Freeze:', manifest['freeze_id'], '| status:', manifest['status'])
print('Transformers:', transformers.__version__, '| tokenizers:', tokenizers.__version__)
print('Output:', OUTPUT)

## 1. Verify freeze checksums and tokenizer fingerprint

The fingerprint confirms the **current** candidate tokenizer matches the freeze. It cannot establish what files were loaded in the original audit session.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(8*1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

for raw in (FREEZE/'SHA256SUMS.txt').read_text().splitlines():
    expected, name = raw.split('  ', 1)
    assert sha256_file(FREEZE/name) == expected, f'Freeze artifact changed: {name}'
print('Freeze manifest/report checksum: PASS')

current_files = {str(p.relative_to(TOKENIZER_DIR)): sha256_file(p)
                 for p in sorted(TOKENIZER_DIR.rglob('*')) if p.is_file()}
current_fingerprint = hashlib.sha256(json.dumps(current_files,sort_keys=True,separators=(',', ':')).encode()).hexdigest()
expected_fingerprint = manifest['candidate_tokenizer']['fingerprint_sha256']
assert current_fingerprint == expected_fingerprint, 'Candidate tokenizer changed since freeze!'
print('Candidate tokenizer matches freeze:', current_fingerprint)
print('Historical identity proven: NO (audit config had tokenizer_path="")')

## 2. Read audit metadata and locate actual chunks

We use filenames stored in `completed`, then locate matching `.jsonl.gz` files under acquisition roots. If your chunks live elsewhere, append their parent directory to `SEARCH_ROOTS`. The discovery intentionally avoids reading all 21,021 compressed files.

In [ ]:
conn = sqlite3.connect(f'file:{AUDIT_DB}?mode=ro', uri=True)
conn.row_factory = sqlite3.Row
assert conn.execute('PRAGMA quick_check').fetchone()[0] == 'ok'
records = [dict(r) for r in conn.execute('SELECT name,job_id,docs,real_tokens,proxy_tokens,compressed_size,mtime_ns FROM completed ORDER BY job_id,name')]
conn.close()
by_job = defaultdict(list)
for r in records: by_job[r['job_id']].append(r)
print('Datasets:', {k:len(v) for k,v in by_job.items()})
assert len(records) == 21021

rng = random.Random(SEED)
sample = []
for job, entries in sorted(by_job.items()):
    sample.extend(rng.sample(entries, min(SAMPLE_CHUNKS_PER_DATASET, len(entries))))
print('Sampled chunks:', len(sample))

# Efficiently search for each sampled filename using glob; no corpus reads here.
def locate_chunk(name):
    for root in SEARCH_ROOTS:
        if not root.exists(): continue
        matches = list(root.rglob(name))
        if matches:
            if len(matches) > 1:
                print('WARNING: multiple paths for', name, 'using', matches[0])
            return matches[0]
    return None

resolved = [(r, locate_chunk(r['name'])) for r in sample]
for r,p in resolved: print(r['job_id'], r['name'], '=>', p or 'NOT FOUND')
missing = [r['name'] for r,p in resolved if p is None]
if missing:
    print('
STOP: add corpus parent directory to SEARCH_ROOTS and rerun this cell. Missing:', len(missing))

## 3. Inspect JSONL schema safely

The audit might count `text`, `content`, or another field, and may insert separators. Inspect one document before recounting. The default below accepts `text` or `content`; **if neither is the original audit field, stop and set `TEXT_FIELD` manually**.

In [ ]:
if missing:
    raise RuntimeError('Missing sampled corpus files; fix SEARCH_ROOTS before proceeding.')
first_path = resolved[0][1]
with gzip.open(first_path, 'rt', encoding='utf-8') as f:
    first_obj = json.loads(next(f))
print('JSON keys:', list(first_obj.keys()))
print('Field types:', {k:type(v).__name__ for k,v in first_obj.items()})
print('Text preview:', repr(str(first_obj.get('text', first_obj.get('content','')))[:240])) )
TEXT_FIELD = 'text' if isinstance(first_obj.get('text'), str) else ('content' if isinstance(first_obj.get('content'),str) else None)
assert TEXT_FIELD, 'Cannot infer text field; inspect JSONL schema and set TEXT_FIELD.'
print('Selected TEXT_FIELD:', TEXT_FIELD)

## 4. Tokenizer recount — **full selected chunks**

This recount is not truncated. It compares counts per chunk against the original SQLite, using the same document text field. It tries three explicit counting conventions: raw text, raw text with special tokens, and raw text plus one EOS token per document. Only an exact match across **all sampled chunks** is evidence that the candidate tokenizer and convention are consistent with the historical audit. A mismatch is a **STOP** for production packing, not permission to rewrite the audit.

Long text warnings from model context are irrelevant for counting; no forward pass is performed.

In [ ]:
tok = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR), use_fast=True, local_files_only=True)
assert tok.is_fast, 'Fast tokenizer required for reproducible pilot speed'
tok.model_max_length = 10**12  # Counting only, not the model context size
print('Tokenizer:', type(tok).__name__, 'vocab:', len(tok), 'EOS:', tok.eos_token_id, 'PAD:', tok.pad_token_id)

recounts=[]
for rec, path in resolved:
    counts = {'raw':0, 'with_special':0, 'raw_plus_eos':0}
    docs = 0
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            obj=json.loads(line)
            value=obj.get(TEXT_FIELD)
            if not isinstance(value,str):
                raise ValueError(f'Missing text in {path.name}, document {docs}')
            ids=tok.encode(value,add_special_tokens=False,truncation=False)
            counts['raw']+=len(ids)
            counts['raw_plus_eos']+=len(ids)+(1 if tok.eos_token_id is not None else 0)
            counts['with_special']+=len(tok.encode(value,add_special_tokens=True,truncation=False))
            docs+=1
    doc_match = docs==rec['docs']
    matched=[mode for mode,val in counts.items() if val==rec['real_tokens']]
    recounts.append({'dataset':rec['job_id'],'chunk':rec['name'],'docs_audit':rec['docs'],'docs_recount':docs,
                     'audit_tokens':rec['real_tokens'],**counts,'matching_modes':matched,'docs_match':doc_match})
    print(rec['job_id'], rec['name'], 'docs:',docs, 'audit:',rec['real_tokens'],
          'raw:',counts['raw'],'special:',counts['with_special'],'+EOS:',counts['raw_plus_eos'],
          'MATCH:', matched or 'NONE')

common_modes=set.intersection(*(set(r['matching_modes']) for r in recounts)) if recounts else set()
all_docs_match=all(r['docs_match'] for r in recounts)
print('
All document counts match:', all_docs_match)
print('Counting modes matching EVERY chunk:', sorted(common_modes))
RECOUNT_PASS = bool(common_modes) and all_docs_match
print('RECOUNT:', 'PASS (candidate consistency only)' if RECOUNT_PASS else 'REVIEW / STOP')

## 5. Pilot split and sequence packing

**Do not execute as a production data builder.** The pilot is capped at `MAX_PILOT_DOCS`, operates on sampled chunks only, and writes new files to `pretraining_prep_pilot_v1`.

Split uses SHA-256 of normalized document text so exact normalized duplicates map to the same split, including across sampled sources. This is not semantic deduplication. The pilot does **not** certify the entire corpus is leakage-free. Token packing is performed **separately** for train and validation and inserts EOS between documents. The final partial block is discarded, and token loss is measured. Model context length remains provisional.

In [ ]:
if not RECOUNT_PASS:
    raise RuntimeError('STOP: recount did not match audit. Investigate tokenizer, text field and audit counting convention first.')
if tok.eos_token_id is None:
    raise RuntimeError('STOP: no EOS token defined; define an explicit document separator policy.')

def normalize_for_split(text):
    return re.sub(r'\s+', ' ', text).strip()

def split_of(text):
    digest=hashlib.sha256(normalize_for_split(text).encode('utf-8')).digest()
    return 'validation' if int.from_bytes(digest[:8], 'big') % 100 < VALIDATION_PERCENT else 'train'

pilot_docs=[]
for rec,path in resolved:
    with gzip.open(path,'rt',encoding='utf-8') as f:
        for line in f:
            if len(pilot_docs)>=MAX_PILOT_DOCS: break
            if not line.strip(): continue
            obj=json.loads(line)
            txt=obj[TEXT_FIELD]
            if not isinstance(txt,str) or not txt.strip(): continue
            pilot_docs.append((rec['job_id'],txt))
    if len(pilot_docs)>=MAX_PILOT_DOCS: break

# Group identical normalized texts into the same split and count repeated docs.
unique_text_hashes=set()
duplicated_in_pilot=0
streams={'train':[],'validation':[]}
doc_counts=Counter()
for job,txt in pilot_docs:
    digest=hashlib.sha256(normalize_for_split(txt).encode('utf-8')).hexdigest()
    if digest in unique_text_hashes: duplicated_in_pilot+=1
    unique_text_hashes.add(digest)
    part=split_of(txt)
    streams[part].extend(tok.encode(txt,add_special_tokens=False,truncation=False))
    streams[part].append(tok.eos_token_id)
    doc_counts[(job,part)]+=1

blocks={part:[tokens[i:i+BLOCK_SIZE] for i in range(0,len(tokens)-BLOCK_SIZE+1,BLOCK_SIZE)]
        for part,tokens in streams.items()}
for part in ('train','validation'):
    print(part, 'documents:',sum(v for (j,p),v in doc_counts.items() if p==part),
          'tokens:',len(streams[part]),'blocks:',len(blocks[part]),
          'discarded_tail:',len(streams[part])%BLOCK_SIZE)
print('Pilot normalized exact duplicate occurrences:', duplicated_in_pilot)
print('Note: a small pilot with 1% validation can have zero validation blocks.')

## 6. Persist pilot evidence (never overwrite)

The pilot stores its inputs and counts, sampled chunk names, token fingerprint, audit matching modes, and reproducibility settings. Binary token sequences are intentionally **not** written: this avoids accidentally treating the bounded pilot as a production dataset. The source freeze remains unchanged.

In [ ]:
report={
 'schema_version':'1.0', 'created_utc':datetime.now(timezone.utc).isoformat(),
 'source_freeze_id':manifest['freeze_id'], 'source_freeze_manifest_sha256':sha256_file(MANIFEST_PATH),
 'candidate_tokenizer_fingerprint':current_fingerprint,
 'historical_tokenizer_identity_verified':False,
 'tokenizer_recount_consistent_with_audit':RECOUNT_PASS,
 'matching_count_modes':sorted(common_modes), 'text_field':TEXT_FIELD,
 'recount_sample':recounts,
 'pilot':{'seed':SEED,'max_docs':MAX_PILOT_DOCS,'sample_chunks_per_dataset':SAMPLE_CHUNKS_PER_DATASET,
          'block_size':BLOCK_SIZE,'validation_percent':VALIDATION_PERCENT,
          'documents':len(pilot_docs),'duplicate_occurrences':duplicated_in_pilot,
          'documents_by_source_split':{f'{job}/{part}':v for (job,part),v in sorted(doc_counts.items())},
          'tokens_by_split':{k:len(v) for k,v in streams.items()},
          'full_blocks_by_split':{k:len(v) for k,v in blocks.items()},
          'discarded_tail_tokens_by_split':{k:len(v)%BLOCK_SIZE for k,v in streams.items()}},
 'environment':{'python':platform.python_version(),'transformers':transformers.__version__,
                'tokenizers':tokenizers.__version__},
 'limitations':['Recount matching is consistency evidence, not historical tokenizer proof.',
                'No full-corpus near-deduplication, license clearance, or leakage audit.',
                'No final model architecture or production context length is fixed.',
                'Pilot uses bounded first documents of sampled chunks; not a representative quality assessment.']
}
if WRITE_PILOT:
    OUTPUT.mkdir(parents=True,exist_ok=True)
    path=OUTPUT/'pilot_report.json'
    with path.open('x',encoding='utf-8') as f:
        json.dump(report,f,indent=2,ensure_ascii=False)
        f.write('\n')
    print('PILOT REPORT CREATED:',path)
else:
    print(json.dumps(report['pilot'],indent=2))
print('SOURCE FREEZE UNCHANGED.')

## 7. Interpretation and next release gate

- **If recount PASS:** candidate tokenizer + one tested counting convention is consistent with all selected chunk totals. Historical identity is **still not proven**; retain provenance pending or document separate historical evidence.
- **If recount FAIL:** inspect audit counting code (field extraction, normalization, special tokens, EOS) before any full preparation.
- **If pilot validation has zero blocks:** increase bounded pilot size or validation share **for testing only**; production validation policy remains to be chosen.
- **Before full pretraining:** freeze architecture and context, decide PT/EN/ES target sampling, licensing and quality filtering, near-deduplication and cross-split leakage policy, and build a resumable streaming tokenizer/packer with per-shard checksums and source-specific held-out evaluation.

The full-corpus materialization should be a **separate, explicitly approved** notebook, not an implicit side effect of validation.